In [1]:
import sys
import os
import gc
import glob
import warnings
import subprocess
import re

import numpy as np
import pandas as pd
import cv2
import tifffile as tiff
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision

warnings.filterwarnings("ignore")

try:
    import segmentation_models_pytorch as smp  # type: ignore
except Exception:
    smp = None



In [2]:
sz = 256  # tile size at network input resolution
reduce = 4  # downscale factor applied before model; later upsample back by this factor
TH = 0.35  # threshold for positive predictions

DATA = "../input/hubmap-kidney-segmentation/test/"

MODELS = [
    f"../input/b4-256-noshifttrain/efficientnet-b2-256-FOLD-{i}-model.pth"
    for i in range(5)
]

df_sample = pd.read_csv("../input/hubmap-kidney-segmentation/sample_submission.csv")

bs = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "efficientnet-b2"
shift = True
minoverlap = 300




In [3]:
def enc2mask(encs, shape):
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for m, enc in enumerate(encs):
        if enc is None or (isinstance(enc, float) and np.isnan(enc)):
            continue
        s = str(enc).split()
        for i in range(len(s) // 2):
            start = int(s[2 * i]) - 1
            length = int(s[2 * i + 1])
            img[start : start + length] = 1 + m
    return img.reshape(shape).T


def mask2enc(mask, n=1):
    pixels = mask.T.flatten()
    encs = []
    for i in range(1, n + 1):
        p = (pixels == i).astype(np.int8)
        if p.sum() == 0:
            encs.append(np.nan)
        else:
            p = np.concatenate([[0], p, [0]])
            runs = np.where(p[1:] != p[:-1])[0] + 1
            runs[1::2] -= runs[::2]
            encs.append(" ".join(str(x) for x in runs))
    return encs


def rle_encode_less_memory(img):
    if img is None or img.size == 0 or img.max() == 0:
        return ""
    pixels = img.T.flatten()
    pixels[0] = 0
    pixels[-1] = 0
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 2
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)




In [4]:
mean = np.array([0.65459856, 0.48386562, 0.69428385], dtype=np.float32)
std = np.array([0.15167958, 0.23584107, 0.13146145], dtype=np.float32)

s_th = 40  # saturation blanking threshold

# CHANGE (score-relevant, minimal): p_th must be in the SAME pixel scale as `s`,
# which is computed AFTER downscaling to (sz x sz). Keep original intent:
# "about 1000 pixels at 256x256", scaled for general sz (at network input scale).
p_th = int(1000 * (sz / 256) ** 2)


def img2tensor(img, dtype: np.dtype = np.float32):
    if img.ndim == 2:
        img = np.expand_dims(img, 2)
    img = np.transpose(img, (2, 0, 1))
    return torch.from_numpy(img.astype(dtype, copy=False))


def make_grid(shape, window=256, min_overlap=32):
    """
    Return array of size (N,4), where N - number of tiles,
    axis represents: x1,x2,y1,y2
    """
    x, y = shape
    nx = x // (window - min_overlap) + 1
    x1 = np.linspace(0, x, num=nx, endpoint=False, dtype=np.int64)
    x1[-1] = x - window
    x2 = (x1 + window).clip(0, x)

    ny = y // (window - min_overlap) + 1
    y1 = np.linspace(0, y, num=ny, endpoint=False, dtype=np.int64)
    y1[-1] = y - window
    y2 = (y1 + window).clip(0, y)

    slices = np.zeros((nx, ny, 4), dtype=np.int64)
    for i in range(nx):
        for j in range(ny):
            slices[i, j] = x1[i], x2[i], y1[j], y2[j]
    return slices.reshape(nx * ny, 4)


def _ensure_hwc_3ch(arr: np.ndarray) -> np.ndarray:
    """Normalize to HWC with exactly 3 channels."""
    mm = arr
    if mm.ndim == 2:
        mm = mm[..., None]
    if mm.ndim == 3 and mm.shape[0] in (3, 4) and mm.shape[-1] not in (3, 4):
        mm = np.moveaxis(mm, 0, -1)
    if mm.shape[-1] < 3:
        mm = np.repeat(mm, 3, axis=-1)
    elif mm.shape[-1] > 3:
        mm = mm[..., :3]
    return mm


def _tiff_is_jpeg_compressed(path: str) -> bool:
    try:
        with tiff.TiffFile(path) as tif:
            page = tif.pages[0]
            try:
                comp_tag = page.tags.get("Compression", None)
                if comp_tag is not None:
                    comp_val = comp_tag.value
                    if isinstance(comp_val, (tuple, list)):
                        comp_val = comp_val[0]
                    if isinstance(comp_val, (int, np.integer)):
                        return int(comp_val) == 7
                    if isinstance(comp_val, str):
                        return "JPEG" in comp_val.upper()
            except Exception:
                pass

            comp = getattr(page, "compression", None)
            if isinstance(comp, (int, np.integer)):
                return int(comp) == 7
            return "JPEG" in str(comp).upper()
    except Exception:
        return False


def _read_tiff_pil(path: str):
    try:
        from PIL import Image

        with Image.open(path) as im:
            im.load()
            arr = np.array(im)
        if arr is None:
            return None
        if arr.ndim == 2:
            return arr
        if arr.ndim == 3 and arr.shape[-1] >= 3:
            arr = arr[..., :3]
            arr = arr[..., ::-1].copy()  # RGB->BGR for downstream cv2 usage
        return arr
    except Exception:
        return None


def _read_tiff_opencv(path: str):
    try:
        cv_img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if cv_img is not None:
            return cv_img
    except Exception:
        pass
    try:
        with open(path, "rb") as f:
            buf = np.frombuffer(f.read(), dtype=np.uint8)
        cv_img = cv2.imdecode(buf, cv2.IMREAD_UNCHANGED)
        return cv_img
    except Exception:
        return None


def _decode_jpeg_tiff_via_tifffile_cv2(path: str):
    try:
        with tiff.TiffFile(path) as tif:
            page = tif.pages[0]

            if getattr(page, "is_tiled", False):
                try:
                    tj = page.tags.get("TileJPEGInterchangeFormat", None)
                    tl = page.tags.get("TileJPEGInterchangeFormatLength", None)
                    if tj is not None and tl is not None:
                        offsets = np.array(tj.value).ravel().tolist()
                        lengths = np.array(tl.value).ravel().tolist()
                        if len(offsets) > 0 and len(offsets) == len(lengths):
                            with open(path, "rb") as f:
                                f.seek(int(offsets[0]))
                                jpeg_bytes = f.read(int(lengths[0]))
                            buf = np.frombuffer(jpeg_bytes, dtype=np.uint8)
                            img = cv2.imdecode(buf, cv2.IMREAD_COLOR)
                            return img
                except Exception:
                    pass

            try:
                sj = page.tags.get("JPEGInterchangeFormat", None)
                sl = page.tags.get("JPEGInterchangeFormatLength", None)
                if sj is not None and sl is not None:
                    offset = int(sj.value)
                    length = int(sl.value)
                    with open(path, "rb") as f:
                        f.seek(offset)
                        jpeg_bytes = f.read(length)
                    buf = np.frombuffer(jpeg_bytes, dtype=np.uint8)
                    img = cv2.imdecode(buf, cv2.IMREAD_COLOR)
                    return img
            except Exception:
                pass

    except Exception:
        return None
    return None


def _resolve_tiff_path(base_dir: str, idx: str) -> str:
    p1 = os.path.join(base_dir, idx + ".tiff")
    if os.path.exists(p1):
        return p1
    p2 = os.path.join(base_dir, idx + ".tif")
    if os.path.exists(p2):
        return p2
    return p1


def _tifftools_convert_to_uncompressed(
    src_path: str, out_dir: str = "/kaggle/working"
) -> str:
    """
    Runtime fallback: rewrite problematic JPEG-compressed TIFFs into a decodeable one.
    """
    os.makedirs(out_dir, exist_ok=True)
    dst_path = os.path.join(
        out_dir, os.path.basename(src_path).replace(".tif", ".tiff")
    )
    dst_path = dst_path.replace(".tiff", "") + "__converted.tiff"
    if os.path.exists(dst_path) and os.path.getsize(dst_path) > 0:
        return dst_path

    tt_local = "../input/hubmap-kidney-segmentation/tifftools"
    tifftools_cmd = tt_local if os.path.exists(tt_local) else "tifftools"
    cmd = [tifftools_cmd, "tiff_set", src_path, dst_path]
    try:
        subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        if os.path.exists(dst_path) and os.path.getsize(dst_path) > 0:
            return dst_path
    except Exception:
        pass
    return src_path


def open_tiff_memmap(path):
    """
    Ensure we can read JPEG-compressed TIFFs without imagecodecs.
    Order: PIL -> OpenCV -> tifffile tag extract + cv2 -> tifftools rewrite -> tifffile.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"TIFF not found: {path}")

    is_jpeg = _tiff_is_jpeg_compressed(path)

    arr = None
    if is_jpeg:
        arr = _read_tiff_pil(path)
        if arr is None:
            arr = _read_tiff_opencv(path)
        if arr is None:
            arr = _decode_jpeg_tiff_via_tifffile_cv2(path)
        if arr is None:
            conv_path = _tifftools_convert_to_uncompressed(path)
            if conv_path != path:
                try:
                    with tiff.TiffFile(conv_path) as tif:
                        arr = tif.asarray()
                except Exception:
                    arr = None
        if arr is None:
            raise ValueError(
                f"Failed to decode JPEG-compressed TIFF without imagecodecs: {path}"
            )
    else:
        try:
            with tiff.TiffFile(path) as tif:
                try:
                    arr = tif.asarray(out="memmap")
                except Exception:
                    arr = tif.asarray()
        except Exception:
            arr = None

        if arr is None:
            arr = _read_tiff_pil(path)
        if arr is None:
            arr = _read_tiff_opencv(path)
        if arr is None:
            raise ValueError(f"Failed to decode TIFF: {path}")

    mm = _ensure_hwc_3ch(arr)
    if mm.dtype != np.uint8:
        mm = np.clip(mm, 0, 255).astype(np.uint8)
    return mm


def read_window(mm, x1, x2, y1, y2):
    w = mm[x1:x2, y1:y2]
    if w.dtype != np.uint8:
        w = np.clip(w, 0, 255).astype(np.uint8)
    return w




In [5]:
class Model_pred_shift:
    def __init__(self, models, dl, tta: bool = False, half: bool = False):
        self.models = models
        self.dl = dl
        self.tta = tta
        self.half = half

    def __iter__(self):
        with torch.no_grad():
            for x, z, y in iter(self.dl):
                if (y >= 0).sum() > 0:  # exclude empty tiles
                    x = x[y >= 0].to(device)
                    z = z[y >= 0]
                    y = y[y >= 0]
                    if self.half:
                        x = x.half()
                    py = None
                    for model in self.models:
                        p = model(x)
                        p = torch.sigmoid(p).detach()
                        py = p if py is None else (py + p)

                    if self.tta:
                        flips = [[-1], [-2], [-2, -1]]
                        for f in flips:
                            xf = torch.flip(x, f)
                            for model in self.models:
                                p = model(xf)
                                p = torch.flip(p, f)
                                py += torch.sigmoid(p).detach()
                        py /= 1 + len(flips)

                    py /= max(1, len(self.models))

                    py = F.interpolate(
                        py, scale_factor=reduce, mode="bilinear", align_corners=False
                    )
                    py = py.permute(0, 2, 3, 1).float().cpu()

                    py = py.squeeze(-1).numpy()
                    z = z.numpy()

                    batch_size = len(py)
                    for i in range(batch_size):
                        yield py[i], z[i], y[i]

    def __len__(self):
        return len(self.dl.dataset)


class HuBMAPDatasetShift(Dataset):
    def __init__(self, idx, sz=sz, reduce=reduce):
        path = _resolve_tiff_path(DATA, idx)
        self.mm = open_tiff_memmap(path)
        self.shape = self.mm.shape[:2]  # (H,W)
        self.reduce = reduce
        self.sz = reduce * sz
        self.mask_grid = make_grid(self.shape, window=self.sz, min_overlap=minoverlap)

    def __len__(self):
        return len(self.mask_grid)

    def __getitem__(self, idx):
        x1, x2, y1, y2 = self.mask_grid[idx]
        img = read_window(self.mm, x1, x2, y1, y2)

        if self.reduce != 1:
            img = cv2.resize(
                img,
                (self.sz // reduce, self.sz // reduce),
                interpolation=cv2.INTER_AREA,
            )

        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        _, s, _ = cv2.split(hsv)
        vertices = torch.tensor([x1, x2, y1, y2], dtype=torch.int64)

        if (s > s_th).sum() <= p_th:
            return img2tensor((img / 255.0 - mean) / std), vertices, -1
        else:
            return img2tensor((img / 255.0 - mean) / std), vertices, idx




In [6]:
class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv1 = ConvBNReLU(in_ch + skip_ch, out_ch, 3, 1, 1)
        self.conv2 = ConvBNReLU(out_ch, out_ch, 3, 1, 1)

    def forward(self, x, skip):
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        if skip is not None:
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(
                    x, size=skip.shape[-2:], mode="bilinear", align_corners=False
                )
            x = torch.cat([x, skip], dim=1)
        return self.conv2(self.conv1(x))


class EfficientNetB2UNet(nn.Module):
    def __init__(self, classes=1):
        super().__init__()
        self.encoder = torchvision.models.efficientnet_b2(weights=None)
        self.features = self.encoder.features

        self._tap_indices = [1, 2, 3, 5, 8]

        with torch.no_grad():
            x = torch.zeros(1, 3, 256, 256)
            skips = []
            for i, layer in enumerate(self.features):
                x = layer(x)
                if i in self._tap_indices:
                    skips.append(x)
            enc_ch = [t.shape[1] for t in skips]

        c1, c2, c3, c4, c5 = enc_ch  # shallow->deep
        self.center = ConvBNReLU(c5, 256, 3, 1, 1)
        self.dec4 = DecoderBlock(256, c4, 256)
        self.dec3 = DecoderBlock(256, c3, 128)
        self.dec2 = DecoderBlock(128, c2, 64)
        self.dec1 = DecoderBlock(64, c1, 32)
        self.head = nn.Conv2d(32, classes, kernel_size=1)

    def forward(self, x):
        skips = []
        for i, layer in enumerate(self.features):
            x = layer(x)
            if i in self._tap_indices:
                skips.append(x)
        s1, s2, s3, s4, s5 = skips  # shallow->deep
        x = self.center(s5)
        x = self.dec4(x, s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)
        return self.head(x)


class FallbackUNet(nn.Module):
    def __init__(self, in_channels=3, classes=1, base=32):
        super().__init__()
        self.inc = nn.Sequential(
            nn.Conv2d(in_channels, base, 3, padding=1, bias=False),
            nn.BatchNorm2d(base),
            nn.ReLU(inplace=True),
            nn.Conv2d(base, base, 3, padding=1, bias=False),
            nn.BatchNorm2d(base),
            nn.ReLU(inplace=True),
        )
        self.down1 = nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(base, base * 2, 3, padding=1, bias=False),
            nn.BatchNorm2d(base * 2),
            nn.ReLU(inplace=True),
        )
        self.down2 = nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(base * 2, base * 4, 3, padding=1, bias=False),
            nn.BatchNorm2d(base * 4),
            nn.ReLU(inplace=True),
        )
        self.down3 = nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(base * 4, base * 8, 3, padding=1, bias=False),
            nn.BatchNorm2d(base * 8),
            nn.ReLU(inplace=True),
        )
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.outc = nn.Conv2d(base * 8, classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x = self.up(x4)
        return self.outc(x)


def build_model():
    if model_name == "efficientnet-b2":
        return EfficientNetB2UNet(classes=1)
    if smp is not None:
        return smp.Unet(model_name, encoder_weights=None, classes=1)
    return FallbackUNet(in_channels=3, classes=1, base=32)


def _find_model_files(requested_paths):
    existing = [p for p in requested_paths if os.path.exists(p)]
    if existing:
        return existing

    candidates = glob.glob("../input/**/*.pth", recursive=True)
    fold_like = []
    for p in candidates:
        b = os.path.basename(p).lower()
        if ("fold" in b) and ("efficientnet" in b or "eff" in b) and ("model" in b):
            fold_like.append(p)
    fold_like = sorted(fold_like)
    if fold_like:
        return fold_like[:5]

    candidates = sorted(candidates)
    return candidates[:5]


MODEL_FILES = _find_model_files(MODELS)
print("Found model files:", MODEL_FILES if MODEL_FILES else "NONE")


def _extract_state_dict(obj):
    if isinstance(obj, dict):
        for k in ("state_dict", "model", "model_state_dict", "net", "weights"):
            if k in obj and isinstance(obj[k], dict):
                return obj[k]
    return obj


def _strip_module_prefix(sd):
    if not isinstance(sd, dict):
        return sd
    if any(k.startswith("module.") for k in sd.keys()):
        return {k[len("module.") :]: v for k, v in sd.items()}
    return sd




Found model files: NONE


In [7]:
models = []
if MODEL_FILES:
    for path in MODEL_FILES:
        raw = torch.load(path, map_location=torch.device("cpu"))
        state_dict = _strip_module_prefix(_extract_state_dict(raw))

        model = build_model()
        loaded = False
        try:
            model.load_state_dict(state_dict, strict=True)
            loaded = True
        except Exception:
            try:
                model.load_state_dict(state_dict, strict=False)
                loaded = True
            except Exception:
                loaded = False

        model.float()
        model.eval()
        model.to(device)
        models.append(model)
        print(f"Loaded: {os.path.basename(path)} (state_dict_loaded={loaded})")

    del raw, state_dict
    gc.collect()
else:
    model = build_model().to(device).eval().float()
    models = [model]
    print(
        "No .pth weights found; using randomly initialized model (submission will be low-scoring)."
    )



No .pth weights found; using randomly initialized model (submission will be low-scoring).


In [8]:
names, preds = [], []

for _, row in df_sample.iterrows():
    idx = row["id"]
    try:
        ds = HuBMAPDatasetShift(idx)
        dl = DataLoader(
            ds, batch_size=bs, pin_memory=True, shuffle=False, num_workers=0
        )
        mp = Model_pred_shift(models, dl)

        mask = np.zeros(ds.shape, dtype=np.uint16)
        for pred, vert, i in iter(mp):
            x1, x2, y1, y2 = vert
            mask[x1:x2, y1:y2] += (pred > TH).astype(np.uint16)

        # CHANGE (score-relevant, minimal): for integer vote masks this is identical to >0.5,
        # but explicitly expresses "any positive vote" and avoids dtype/edge surprises.
        mask = (mask > 0).astype(np.uint8)

        rle = rle_encode_less_memory(mask)
    except Exception as e:
        print(f"[WARN] Failed on id={idx}: {repr(e)}; writing empty mask.")
        rle = ""

    names.append(idx)
    preds.append(rle)

    try:
        del mask, ds, dl
    except Exception:
        pass
    gc.collect()



[WARN] Failed on id=8242609fa: ValueError('Failed to decode JPEG-compressed TIFF without imagecodecs: ../input/hubmap-kidney-segmentation/test/8242609fa.tiff'); writing empty mask.
[WARN] Failed on id=0486052bb: ValueError('operands could not be broadcast together with shapes (1024,1024) (512,512) (1024,1024) '); writing empty mask.
[WARN] Failed on id=095bf7a1f: ValueError('operands could not be broadcast together with shapes (1024,1024) (512,512) (1024,1024) '); writing empty mask.


In [9]:
sub = pd.DataFrame({"id": names, "predicted": preds})

sub = df_sample[["id"]].merge(sub, on="id", how="left")
sub["predicted"] = sub["predicted"].fillna("")

sub.to_csv("submission.csv", index=False)
print(sub.head())
print("Wrote submission.csv with shape:", sub.shape)
print("Saved to:", os.path.abspath("submission.csv"))

          id predicted
0  8242609fa          
1  0486052bb          
2  095bf7a1f          
Wrote submission.csv with shape: (3, 2)
Saved to: /kaggle/working/submission.csv
